# XL Complex Structure — Data Postprocessing

本 notebook 用于对 AlphaFold/ColabFold 输出的结构文件进行后处理，包括：

1. **CIF → PDB 格式转换**（BioPython）
2. **多 PDB 合并与链删除**（PyMOL open-source）

运行环境：conda `xl_complex_structure`

## 1. CIF → PDB 格式转换

使用 BioPython 的 `MMCIFParser` 读取 mmCIF 文件，再用 `PDBIO` 写出为标准 PDB 格式。

In [4]:
# rewrite cif to pdb
from Bio.PDB import MMCIFParser, PDBIO

def cif_to_pdb(cif_path, pdb_path):
    """
    使用 BioPython 将 mmCIF 转为 PDB
    """
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("structure", cif_path)

    io = PDBIO()
    io.set_structure(structure)
    io.save(pdb_path)


cif_to_pdb(r"N:\08_NK_structure_prediction\data\LRBAandSNARE\assembled_complex\pairs\lrba_0\lrba_0_model.cif", 
                       r"N:\08_NK_structure_prediction\data\LRBAandSNARE\assembled_complex\pairs\lrba_0\lrba_0_model.pdb")

## 2. PDB 合并与链删除（PyMOL）

`merge_pdb_remove_chain_pymol` 的工作流程：

1. 将两个 PDB 加载到 PyMOL session 中
2. 以 `pdb1` 的 `align_chain1` 为参考，把 `pdb2` 的 `align_chain2` 对齐上去（`cmd.align`）
3. 从 `pdb2` 中删除对齐所用的冗余链（`remove_chain`）
4. 将两个对象合并为一个新对象并保存为 PDB

> **依赖**：需要 `pymol-open-source`（`conda install -c conda-forge pymol-open-source`）

In [2]:
from pymol import cmd, stored

def merge_pdb_remove_chain_pymol(
    pdb1_path, pdb2_path,
    align_chain1, align_chain2,
    remove_chain, output_path
):
    cmd.reinitialize()

    cmd.load(pdb1_path, "pdb1")
    cmd.load(pdb2_path, "pdb2")

    # 用 super，更接近 GUI 行为
    cmd.align(
        f"pdb2 and chain {align_chain2}",
        f"pdb1 and chain {align_chain1}"
    )

    if remove_chain is not None:
        cmd.remove(f"pdb2 and chain {remove_chain}")

    cmd.create("merged", "pdb1 or pdb2")
    cmd.save(output_path, "merged")



### 示例：将 Rab11b/EXOC6 亚复合体拼入 Exocyst 全复合体

- `pdb1`：已有的 Exocyst 最优复合体（`best_complex.pdb`），其 chain F = EXOC6
- `pdb2`：新预测的 Rab11b–EXOC6 二聚体（`rab11b_exoc6_0_model.pdb`），其 chain F = EXOC6
- 对齐后删去 `pdb2` 中的 chain F（冗余），保留 Rab11b 链
- 输出：合并后包含 Rab11b 的完整复合体

In [6]:
merge_pdb_remove_chain_pymol(
    pdb1_path=r"N:\08_NK_structure_prediction\data\LRBAandSNARE\assembled_complex\pairs\lrba_0\lrba_0_model.pdb",
    pdb2_path=r"N:\08_NK_structure_prediction\data\LRBAandSNARE\assembled_complex\pairs\lrba_stx12_0_AC\lrba_stx12_0_AC_A-lrba_stx12_0_AC_C.pdb",
    align_chain1="A",
    align_chain2="A",
    remove_chain="A",   # 删除对齐链中的哪一个
    output_path=r"N:\08_NK_structure_prediction\data\LRBAandSNARE\assembled_complex\output\lrba_stx12_complete.pdb"
)